# ZelusBench: Selective Attention

**Does the model get distracted by irrelevant but salient information?**

Varies distractor density while randomizing ALL background conditions
(chain depth, DAG structure, transforms, dimensionality, point types).

- Levels: CLEAN (0:1), LOW (1:1), HIGH (3:1), EXTREME (10:1)
- 10 seeds per level = 40 scenarios, ~120 queries

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import ScenarioConfig, DistractorLevel
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd))
            for qd, rp in zip(query_dicts, parts)]


DISTRACTOR_LEVELS = [
    (0, DistractorLevel.CLEAN),
    (1, DistractorLevel.LOW),
    (3, DistractorLevel.HIGH),
    (10, DistractorLevel.EXTREME),
]
SEEDS = 10
TOTAL = len(DISTRACTOR_LEVELS) * SEEDS

print(f"ZelusBench Selective Attention")
print(f"Ratios: {[r for r, _ in DISTRACTOR_LEVELS]} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_selective_attention")
def zelusbench_selective_attention(llm) -> tuple[float, float]:
    """Selective attention: accuracy vs distractor density (randomized background).
    Returns (mean_accuracy, std_dev)."""

    _start = time.time()
    print(f"Running {TOTAL} scenarios across {len(DISTRACTOR_LEVELS)} distractor levels...")
    print("=" * 60)

    all_scores = []
    level_scores = {}
    scenario_num = 0

    for ratio, level in DISTRACTOR_LEVELS:
        level_scores[ratio] = []
        for seed in range(SEEDS):
            scenario_num += 1
            unique_seed = ratio * 1000 + seed
            rng = random.Random(unique_seed)
            cfg = ScenarioConfig.randomize_except(rng, pinned={
                "distractor_level": level,
                "num_queries": 3,
                "seed": unique_seed,
            })
            gen = ScenarioGenerator(cfg)
            scenario = gen.generate(f"selective_r{ratio}_s{seed}")

            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            all_scores.extend(scores)

            avg = float(np.mean([s.score for s in scores]))
            level_scores[ratio].append(avg)

            q_depths = [s.chain_depth for s in scores]
            tiers = [s.tier.name for s in scores]
            bg = f"dim={cfg.dim} dag={cfg.dag_structure.name} depth={cfg.min_chain_depth}-{cfg.max_chain_depth} tx={cfg.transform_density.name}"
            print(f"  [{scenario_num}/{TOTAL}] ratio={ratio}:1 seed={seed} "
                  f"avg={avg:.2f} q_depths={q_depths} tiers={tiers} | {bg}")

        level_avg = float(np.mean(level_scores[ratio]))
        print(f"  >> Ratio {ratio}:1 mean: {level_avg:.3f}")

    print("\n" + "=" * 60)
    print("RESULTS BY DISTRACTOR RATIO:")
    clean_acc = float(np.mean(level_scores.get(0, [0])))
    for ratio, _ in DISTRACTOR_LEVELS:
        avg = float(np.mean(level_scores[ratio]))
        tax = clean_acc - avg if clean_acc > 0 else 0
        print(f"  ratio={ratio:2d}:1  accuracy={avg:.3f}  distractor_tax={tax:+.3f}")
        kbench.assertions.assert_true(
            avg >= 0,
            expectation=f"Ratio {ratio}:1: model should produce valid answers (got {avg:.3f})"
        )

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start

    print(f"\nOverall: {overall:.3f} +/- {std:.3f}")
    print(f"Total queries: {len(all_scores)}")
    print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    return overall, std


zelusbench_selective_attention.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_selective_attention